In [ ]:
!pip -q install chromadb langchain_google_genai langchain langchain-community langchain-text-splitters tiktoken
!pip show chromadb
!wget -q https://www.dropbox.com/s/vs6ocyvpzzncvwh/new_articles.zip
!unzip -q -o new_articles.zip -d new_articles
!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community

In [ ]:
import os
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma

In [ ]:
# GOOGLE_API_KEY = 'your-google-api-key-here'
# os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
# TODO: Set your Google API key as an environment variable or replace above line

In [ ]:
loader = DirectoryLoader("/content/new_articles/", glob="./*.txt", loader_cls=TextLoader)
document = loader.load()
document

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
text = text_splitter.split_documents(document)
text

In [ ]:
len(text), text[1], text[2]

In [ ]:
# import chromadb
#
# client = chromadb.CloudClient(
#   api_key='your-chromadb-cloud-api-key-here',
#   tenant='your-tenant-id-here',
#   database='your-database-name-here'
# )
#
# TODO: Set your ChromaDB Cloud credentials above or use local ChromaDB instead

In [ ]:
persist_directory = 'db'
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vectordb = Chroma.from_documents(documents=text, embedding=embedding, persist_directory=persist_directory)

vectordb = None
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

In [ ]:
retriever = vectordb.as_retriever()
docs = retriever.get_relevant_documents("How much money did Microsoft raise?")
docs , len(docs)

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k": 2})
retriever.search_type, retriever.search_kwargs

In [ ]:
from langchain.chains import RetrievalQA

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)
llm

In [ ]:
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=True)

def process_llm_response(llm_response):
    print(llm_response['result'])
    print('\n\nSources:')
    for source in llm_response["source_documents"]:
        print(source.metadata['source'])

query = "How much money did Microsoft raise?"
llm_response = qa_chain(query)
process_llm_response(llm_response)

In [ ]:
!zip -r db.zip ./db

In [ ]:
vectordb.delete_collection()

In [ ]:
!rm -rf db/

In [ ]:
!unzip db.zip

# **Current setup**

In [ ]:
from google import genai
from google.generativeai import types
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from google.colab import userdata

api_key = userdata.get('GOOGLE_API_KEY')

if not api_key:
  raise RuntimeError(
      'Please set GOOGLE_API_KEY in the notebook settings.'
      'Add it via colab -> Secrets -> enables Notebook access.'
  )

client = genai.Client(api_key=api_key)

In [ ]:
result = client.models.embed_content(
    model='text-embedding-004',
    contents=[doc.page_content for doc in text],
    config={
        "task_type": "RETRIEVAL_DOCUMENT",
        "output_dimensionality": 768
    }
)

for i, embedding_obj in enumerate(result.embeddings):
  embedding = embedding_obj.values
  print("Text:", text[i].page_content[:100] + "...")
  print("Embeddings length:", len(embedding))
  print("-" * 50)

In [ ]:
# import chromadb
#
# client = chromadb.CloudClient(
#   api_key='your-chromadb-cloud-api-key-here',
#   tenant='your-tenant-id-here',
#   database='your-database-name-here'
# )
#
# collection = client.get_or_create_collection(name="your-collection-name-here")
#
# TODO: Set your ChromaDB Cloud credentials above or use local ChromaDB instead

In [ ]:
embeddings = []
ids = []

for i , embedding_obj in enumerate(result.embeddings):
  embedding = embedding_obj.values
  embeddings.append(embedding)
  ids.append(f"id_{i}")

collection.add(ids=ids, documents=[doc.page_content for doc in text], embeddings=embeddings)

In [ ]:
query = "programming"
query_embedding_result = client.models.embed_content(
    model='gemini-embedding-001',
    contents=[query],
    config=types.EmbeddingModelConfig(
        task_type="SEMANTIC_SIMILARITY",
        outputDimensionality=768
    )
)
query_embedding = query_embedding_result.embeddings[0].values

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=1
)

print(results["documents"])

In [ ]:
doc1 = Document(
  page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
   metadata={"team": "Royal Challengers Bangalore"}
)
doc2 = Document(
  page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  metadata={"team": "Mumbai Indians"}
)
doc3 = Document(
  page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
  metadata={"team": "Chennai Super Kings"}
)
doc4 = Document(
  page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
  metadata={"team": "Mumbai Indians"}
)
doc5 = Document(
  page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
  metadata={"team": "Chennai Super Kings"}
)

In [ ]:
docs = [doc1, doc2, doc3, doc4, doc5]
vector_store = Chroma(
  embedding_function=OpenAIEmbeddings(),
  persist_directory="./my_chroma_db",
  collection_name="my_collection"
)

vector_store.add_documents(docs)
vector_store.get(include=['embeddings','documents', 'metadatas'])

In [ ]:
vector_store.similarity_search(
  query='Who among these are a bowler?,
  k=2
)

In [ ]:
vector_store.similarity_search_with_score(
  query='Who among these are a bowler?',
  k=2
)

In [ ]:
vector_store.similarity_search_with_score(
  query="",
  filter={"team": "Chennai Super Kings"}
)

In [ ]:
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)

In [ ]:
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [ ]:
vector_store.get(include=['embeddings','documents', 'metadatas'])